# Data loaders

> Data loading functionalities

In [ ]:
#| default_exp data.load

In [ ]:
#| hide
from nbdev.showdoc import *
from nbdev.cli import *

In [ ]:
#| export
from fastai.vision.all import *
from fastai.data.all import *
from pathlib import Path
import pandas as pd
#import numpy as np
#from typing import Tuple, Dict, Callable

## Loaders

In [ ]:
#| export
def get_spectra_files(path):
    "Return paths of spectra `.csv` files in path"
    return [f for f in path.ls() if re.match('\d', f.name)]

In [ ]:
path = Path('../_data/kssl-mirs')
fnames = path.ls(); fnames

(#44101) [Path('../_data/kssl-mirs/180338'),Path('../_data/kssl-mirs/172221'),Path('../_data/kssl-mirs/177753'),Path('../_data/kssl-mirs/184798'),Path('../_data/kssl-mirs/53759'),Path('../_data/kssl-mirs/74947'),Path('../_data/kssl-mirs/176681'),Path('../_data/kssl-mirs/1855'),Path('../_data/kssl-mirs/175004'),Path('../_data/kssl-mirs/34499')...]

In [ ]:
get_spectra_files(fnames[0])

[Path('../_data/kssl-mirs/180338/260043XS04.csv'),
 Path('../_data/kssl-mirs/180338/260043XS01.csv'),
 Path('../_data/kssl-mirs/180338/260043XS02.csv'),
 Path('../_data/kssl-mirs/180338/260043XS03.csv')]

In [ ]:
#| export
class SpectraSetup(Transform):
    "Transform that converts to numpy arrays"
    def __init__(self):
        pass
    def encodes(self, fnames):
        n = pd.read_csv(fnames[0]).shape[0]
        m = len(fnames)
        x = np.empty((m,n))
        for i, fname in enumerate(fnames):
            x[i,:] = pd.read_csv(fname)['absorbance'].values
        return tensor(x)

In [ ]:
#| export
def get_target(path, analytes=None):
    path_target = [f for f in path.ls() if re.match('target', f.name)][0]
    df = pd.read_csv(path_target)
    if analytes:
        df = df[df.analyte.isin(analytes)]
    return df['value'].values

In [ ]:
df = get_target(fnames[0]); df

array([ 2.8077302, 22.033033 , 13.7791402])

In [ ]:
#| export
def SpectraBlock():
    return TransformBlock(type_tfms=SpectraSetup())

In [ ]:
#| export
def random_weighted_avg(x):
    "Transform spectra replicates taking their random weighted averages"
    n = len(x)
    def weights(n):
        weights = torch.rand(n)
        return (weights/weights.sum()).unsqueeze(dim=0)
    return torch.matmul(weights(n), x)

In [ ]:
dblock = DataBlock(
    blocks=(SpectraBlock, RegressionBlock),
    get_x=get_spectra_files,
    get_y=partial(get_target, analytes=[725]),
    splitter=RandomSplitter(valid_pct=0.1, seed=42),
    item_tfms=[Transform(random_weighted_avg)]
)

In [ ]:
dsets = dblock.datasets(fnames)
dsets.train[0]

(tensor([[0.2258, 0.2260, 0.2263,  ..., 1.5891, 1.5789, 1.5701],
         [0.2639, 0.2641, 0.2644,  ..., 1.6350, 1.6246, 1.6143],
         [0.2816, 0.2819, 0.2821,  ..., 1.5537, 1.5427, 1.5320],
         [0.3072, 0.3074, 0.3076,  ..., 1.6325, 1.6217, 1.6133]]),
 tensor([0.2244]))

In [ ]:
dls = dblock.dataloaders(fnames, bs=16)

In [ ]:
dls.train.one_batch()

(tensor([[[0.2538, 0.2541, 0.2545,  ..., 1.5133, 1.5094, 1.5057]],
 
         [[0.4877, 0.4883, 0.4889,  ..., 1.6518, 1.6391, 1.6265]],
 
         [[0.1398, 0.1403, 0.1408,  ..., 1.3812, 1.3666, 1.3552]],
 
         ...,
 
         [[0.1906, 0.1908, 0.1909,  ..., 1.4994, 1.5049, 1.5104]],
 
         [[0.3844, 0.3844, 0.3845,  ..., 1.5356, 1.5320, 1.5228]],
 
         [[0.0872, 0.0876, 0.0880,  ..., 1.4735, 1.4687, 1.4652]]]),
 tensor([[0.2495],
         [0.4573],
         [0.5403],
         [1.4658],
         [0.0000],
         [0.0904],
         [0.2172],
         [0.0000],
         [0.0224],
         [0.3851],
         [2.0468],
         [0.2042],
         [0.5193],
         [0.1786],
         [0.5414],
         [0.2484]]))

In [ ]:
dls.valid.one_batch()

(tensor([[[0.3382, 0.3389, 0.3395,  ..., 1.6828, 1.6655, 1.6483]],
 
         [[1.0204, 1.0201, 1.0198,  ..., 1.3171, 1.3157, 1.3139]],
 
         [[0.2609, 0.2611, 0.2613,  ..., 1.5149, 1.5141, 1.5127]],
 
         ...,
 
         [[0.2286, 0.2288, 0.2290,  ..., 1.6269, 1.6229, 1.6186]],
 
         [[0.1504, 0.1508, 0.1512,  ..., 1.3450, 1.3361, 1.3288]],
 
         [[0.1300, 0.1301, 0.1301,  ..., 1.3425, 1.3299, 1.3182]]]),
 tensor([[0.4968],
         [0.3443],
         [0.0000],
         [0.0000],
         [0.9650],
         [0.6087],
         [1.3657],
         [0.0461],
         [1.0763],
         [1.4860],
         [0.7453],
         [0.0448],
         [2.7428],
         [0.1603],
         [0.6544],
         [0.5604]]))